In [43]:
import numpy as np
import  pandas as pd
import warnings

warnings.filterwarnings('ignore')


In [44]:
df = pd.read_csv('qoute_dataset.csv')

In [45]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [46]:
quotes = df['quote']
quotes.head()

0    “The world as we have created it is a process ...
1    “It is our choices, Harry, that show what we t...
2    “There are only two ways to live your life. On...
3    “The person, be it gentleman or lady, who has ...
4    “Imperfection is beauty, madness is genius and...
Name: quote, dtype: str

In [47]:
quotes[0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [48]:
quotes = quotes.str.lower()

In [49]:
quotes[0]

'“the world as we have created it is a process of our thinking. it cannot be changed without changing our thinking.”'

In [50]:
import string
translator = str.maketrans('','',string.punctuation)
quotes = quotes.apply(lambda x : x.translate(translator))

In [51]:
quotes[0]

'“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”'

In [52]:
from tensorflow.keras.preprocessing.text import Tokenizer
vocab_size = 10000

tokenizer = Tokenizer(num_words= vocab_size)
tokenizer.fit_on_texts(quotes)

In [53]:
word_index = tokenizer.word_index
print(len(word_index))

8978


In [54]:
sentences = tokenizer.texts_to_sequences(quotes)

In [55]:
X = []
y = []

for seq in sentences:
    for i in range(1, len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]
        X.append(input_seq)
        y.append(output_seq)


In [56]:
len(X)

85271

In [57]:
len(y)

85271

In [58]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [59]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_pad = pad_sequences(X, maxlen = max_len, padding= 'pre')

In [60]:
X_pad

array([[   0,    0,    0, ...,    0,    0,  713],
       [   0,    0,    0, ...,    0,  713,   62],
       [   0,    0,    0, ...,  713,   62,   29],
       ...,
       [   0,    0,    0, ...,    9,   19, 1125],
       [   0,    0,    0, ...,   19, 1125,    3],
       [   0,    0,    0, ..., 1125,    3,  169]],
      shape=(85271, 745), dtype=int32)

In [61]:
y = np.array(y)

In [62]:
X_pad.shape

(85271, 745)

In [63]:
from tensorflow.keras.utils import to_categorical
y_onehot = to_categorical(y, num_classes= vocab_size )

In [64]:
y_onehot.shape

(85271, 10000)

In [65]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [66]:
embedded_dim = 50
rnn_units = 128


In [67]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedded_dim, input_length=max_len)
)
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))

In [68]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [69]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [70]:
epochs = 100
batch_size = 128


In [71]:
from tensorflow.keras.models import load_model
lstm_model = load_model('lstm_model.h5')

In [72]:
lstm_model.save('lstm_model.h5')

In [73]:
index_to_word = {}
for word,index in word_index.items():
    index_to_word[index] = word

In [74]:
def predictor(model,text,tokenizer,max_len):

  text = text.lower()
  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')

  pred = model.predict(seq,verbose = 0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]

In [75]:
seed_text = "what are you"
next_word = predictor(lstm_model,seed_text,tokenizer,max_len)
print(next_word)

lurking


In [76]:
def generate_text(model,seed_text,num_words,tokenizer,max_len):
  text = seed_text
  for i in range(num_words):
    next_word = predictor(model,text,tokenizer,max_len)
    if next_word == "":
      break
    text += ' ' + next_word
  return text

In [77]:
seed_text = 'life is a'
generate_text = generate_text(lstm_model,seed_text,10,tokenizer,max_len)
print(generate_text)

life is a series of natural and spontaneous changes dont resist them that


In [78]:
import pickle

with open('tokenizer.pkl','wb') as f:
    pickle.dump(tokenizer,f)

In [79]:
with open('max_len.pkl', 'wb') as f:
    pickle.dump(max_len,f)